# So sánh tracker thuần: ByteTrack, OC-SORT (và + nối track)

**Chỉ cần CPU** (Runtime → Change runtime type → CPU là đủ), rồi **Run all**. Dùng lại hộp YOLO11x đã lưu trên Drive (`outputs/dets_yolo`), không cần tải ảnh.

**Các tracker** (cùng hộp, cùng nhãn, cùng cách chấm):
- `bytetrack`: ByteTrack của đề tài (baseline mọi thí nghiệm trước).
- `ocsort`: OC-SORT mã gốc của tác giả (CVPR 2023), tham số mặc định của repo.
- `ocsort_byte`: OC-SORT bật thêm lượt ghép BYTE cho hộp điểm thấp.
- `…+link`: cùng tracker đó cộng bước nối track (cách 3, không SAM 3), mô hình học trên chính track của tracker đó, ngoài tập theo video.

**Chỉ số:** IDSW (và ba kiểu lỗi), IDF1, MOTA, HOTA / DetA / AssA (TrackEval), tốc độ tracker và tốc độ cả pipeline (cộng thời gian YOLO11x đã đo: 1.776 giây cho 19.480 khung).

**Thời gian:** khoảng 20–30 phút. Kết quả trong `rad-work/outputs/trackers_yolo/`: `summary.md`, `summary.csv`, `per_seq.csv`, `tracks/`.

MOTIP không có trong bảng: trọng số công bố chỉ cho DanceTrack, SportsMOT (người) và BFT (chim), không phát hiện được xe; huấn luyện lại trên FastTracker vượt quá khả năng Colab T4.

## 1. Cấu hình, mã nguồn, thư viện

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
CODE = '/content/rad_mot'
WORK = '/content/drive/MyDrive/Saudh/computer_vision/rad-work'
DS   = '/content/datasets/FastTracker'
DATA = f'{DS}/train'
DETS = f'{WORK}/outputs/dets_yolo'
OUT  = f'{WORK}/outputs/trackers_yolo'
n_dets = len([f for f in os.listdir(DETS) if f.endswith('.txt')]) if os.path.isdir(DETS) else 0
print('hộp YOLO trên Drive:', n_dets, '/ 12')
assert n_dets == 12, 'Chưa đủ hộp YOLO: chạy giai đoạn 1 của colab_link_yolo.ipynb trước.'
!rm -rf $CODE && git clone -q https://github.com/dien2810/rad_mot $CODE
%cd $CODE
!pip -q install -r requirements.txt huggingface_hub filterpy
if not os.path.isdir('/content/OC_SORT'):
    !git clone -q https://github.com/noahcao/OC_SORT /content/OC_SORT
!git -C /content/OC_SORT log -1 --format='OC-SORT: commit %h (%cd)'
os.makedirs(f'{CODE}/rad/link', exist_ok=True)

## 2. Mã mới (nhúng trong notebook)

In [ ]:
%%writefile /content/rad_mot/rad/link/__init__.py
from rad.link.linking import (LINK_FEATURES, LinkParams, apply_links, assign, label_links,  # noqa: F401
                              link_candidates, verify_pair)

In [ ]:
%%writefile /content/rad_mot/rad/link/linking.py
"""Learned tracklet linking (method 3) and VOS verification of uncertain links (method 4).

A fragment is an object the tracker loses as track A and restarts later as a new track C.
Fixing it needs no mask propagation: C (and whatever follows it) is relabelled as A.
For every pair (A ended, C born soon after, near where A was heading) a classifier gives
p = P(A and C are the same object), learned from GT on other videos, and

  * p >= tau_high            -> merge directly, no SAM call (method 3);
  * tau_low <= p < tau_high  -> ask a VOS model: seed A's last clean box, propagate to C's
                                start, merge only if the mask lands on C (method 4);
  * otherwise                -> leave the tracks as they are.

Merges are one-to-one (Hungarian on log-odds): an ended track gets at most one successor
and a new track at most one predecessor, so merged tracks never overlap in time.
"""
from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

from rad.boxes import iou_matrix
from rad.io.mot import F, ID, X, Y, W, H, S, C
from rad.sam.masks import ioma

LINK_FEATURES = [
    "dt", "dt_s", "dist_ext", "dist_last", "iou_last", "iou_ext", "log_size_ratio", "log_aspect_ratio",
    "speed_a", "speed_c", "vel_cos", "vel_diff", "len_a", "len_c", "score_a", "score_c",
    "a_end_border", "c_start_border", "same_cls", "n_a_for_c", "n_c_for_a", "rank_for_c", "rank_for_a",
    "density",
]


@dataclass
class LinkParams:
    max_gap_s: float = 3.0       # seconds between A's last frame and C's first frame
    radius: float = 5.0          # keep pairs whose distance (extrapolated or last) <= radius * size of A
    tail: int = 10               # frames of A's end used for its velocity
    head: int = 10               # frames of C's start used for its velocity
    border: float = 0.01         # a box within this fraction of the image size touches the border
    label_k: int = 10            # matched frames used to read the GT identity of a track end/start
    # verification (method 4)
    seed_lookback: int = 15      # look this many frames back from A's end for a clean seed box
    seed_iso_iou: float = 0.3    # a seed box overlapping another track more than this is not clean
    check_len: int = 5           # frames of C checked against the mask
    max_verify_len: int = 150    # longest propagation allowed for one verification
    confirm_ioma: float = 0.5    # median share of the mask inside C's box needed to confirm


def _xyxy(r: np.ndarray) -> np.ndarray:
    r = np.atleast_2d(r)
    return np.stack([r[:, X], r[:, Y], r[:, X] + r[:, W], r[:, Y] + r[:, H]], 1)


def _slope(t: np.ndarray, v: np.ndarray) -> float:
    return float(np.polyfit(t, v, 1)[0]) if len(t) > 1 and np.ptp(t) > 0 else 0.0


def track_summary(tracks: np.ndarray, size: tuple, p: LinkParams) -> pd.DataFrame:
    """One row per track: first/last box, velocities at both ends, length, score, class, border flags."""
    width, height = size
    rows = []
    order = np.lexsort((tracks[:, F], tracks[:, ID]))
    t = tracks[order]
    ids, starts = np.unique(t[:, ID].astype(int), return_index=True)
    bounds = list(starts) + [len(t)]
    for k, tid in enumerate(ids):
        r = t[bounds[k]:bounds[k + 1]]
        cx, cy = r[:, X] + r[:, W] / 2, r[:, Y] + r[:, H] / 2
        tl, hd = r[-p.tail:], r[:p.head]
        tcx, tcy = cx[-p.tail:], cy[-p.tail:]
        hcx, hcy = cx[:p.head], cy[:p.head]
        first, last = r[0], r[-1]

        def at_border(b):
            mx, my = p.border * width, p.border * height
            return float(b[X] <= mx or b[Y] <= my or b[X] + b[W] >= width - mx or b[Y] + b[H] >= height - my)

        cls = r[:, C][r[:, C] >= 0]
        rows.append(dict(
            id=int(tid), start=int(first[F]), end=int(last[F]), length=len(r),
            first_x=first[X], first_y=first[Y], first_w=first[W], first_h=first[H],
            last_x=last[X], last_y=last[Y], last_w=last[W], last_h=last[H],
            vx_end=_slope(tl[:, F], tcx), vy_end=_slope(tl[:, F], tcy),
            vx_start=_slope(hd[:, F], hcx), vy_start=_slope(hd[:, F], hcy),
            score_end=float(np.mean(tl[:, S])), score_start=float(np.mean(hd[:, S])),
            cls=int(pd.Series(cls).mode().iloc[0]) if len(cls) else -1,
            end_border=at_border(last), start_border=at_border(first)))
    return pd.DataFrame(rows).set_index("id") if rows else pd.DataFrame()


def link_candidates(tracks: np.ndarray, size: tuple, fps: float, p: LinkParams | None = None) -> pd.DataFrame:
    """All (A ended, C born) pairs that could be the same object, with LINK_FEATURES."""
    p = p or LinkParams()
    cols = ["a", "c", "a_end", "c_start"] + LINK_FEATURES
    if len(tracks) == 0:
        return pd.DataFrame(columns=cols)
    fps = float(fps or 30.0)
    max_gap = max(1, int(round(p.max_gap_s * fps)))
    ts = track_summary(tracks, size, p)
    first_frame = int(tracks[:, F].min())
    dens = pd.Series(tracks[:, F]).value_counts()
    rows = []
    for cid, c in ts[ts.start > first_frame].iterrows():
        a_all = ts[(ts.end < c.start) & (ts.end >= c.start - max_gap) & (ts.index != cid)]
        if len(a_all) == 0:
            continue
        dt = (c.start - a_all.end).to_numpy(float)
        size_a = np.sqrt(a_all.last_w * a_all.last_h).clip(lower=1.0).to_numpy()
        acx = (a_all.last_x + a_all.last_w / 2).to_numpy()
        acy = (a_all.last_y + a_all.last_h / 2).to_numpy()
        ex, ey = acx + a_all.vx_end.to_numpy() * dt, acy + a_all.vy_end.to_numpy() * dt
        ccx, ccy = c.first_x + c.first_w / 2, c.first_y + c.first_h / 2
        d_ext = np.hypot(ex - ccx, ey - ccy) / size_a
        d_last = np.hypot(acx - ccx, acy - ccy) / size_a
        keep = (d_ext <= p.radius) | (d_last <= p.radius)
        if not keep.any():
            continue
        cbox = np.array([[c.first_x, c.first_y, c.first_x + c.first_w, c.first_y + c.first_h]])
        for j in np.flatnonzero(keep):
            a = a_all.iloc[j]
            abox = np.array([[a.last_x, a.last_y, a.last_x + a.last_w, a.last_y + a.last_h]])
            shift = np.array([[a.vx_end * dt[j], a.vy_end * dt[j]] * 2])
            va, vc = np.array([a.vx_end, a.vy_end]), np.array([c.vx_start, c.vy_start])
            na, nc = np.linalg.norm(va), np.linalg.norm(vc)
            rows.append(dict(
                a=int(a.name), c=int(cid), a_end=int(a.end), c_start=int(c.start),
                dt=float(dt[j]), dt_s=float(dt[j] / fps), dist_ext=float(d_ext[j]), dist_last=float(d_last[j]),
                iou_last=float(iou_matrix(abox, cbox)[0, 0]), iou_ext=float(iou_matrix(abox + shift, cbox)[0, 0]),
                log_size_ratio=float(np.log(max(np.sqrt(c.first_w * c.first_h), 1.0) / size_a[j])),
                log_aspect_ratio=float(np.log(max(c.first_w / max(c.first_h, 1.0), 1e-3)
                                              / max(a.last_w / max(a.last_h, 1.0), 1e-3))),
                speed_a=float(na / size_a[j]), speed_c=float(nc / size_a[j]),
                vel_cos=float(va @ vc / (na * nc)) if na > 1e-6 and nc > 1e-6 else 0.0,
                vel_diff=float(np.linalg.norm(va - vc) / size_a[j]),
                len_a=float(a.length), len_c=float(c.length), score_a=float(a.score_end), score_c=float(c.score_start),
                a_end_border=float(a.end_border), c_start_border=float(c.start_border),
                same_cls=float(a.cls == c.cls), density=float(dens.get(int(c.start), 0))))
    out = pd.DataFrame(rows, columns=cols)
    if len(out):
        out["n_a_for_c"] = out.groupby("c").a.transform("size").astype(float)
        out["n_c_for_a"] = out.groupby("a").c.transform("size").astype(float)
        out["rank_for_c"] = out.groupby("c").dist_ext.rank(method="first").astype(float)
        out["rank_for_a"] = out.groupby("a").dist_ext.rank(method="first").astype(float)
    return out


def track_identities(matches: pd.DataFrame, k: int) -> pd.DataFrame:
    """GT identity at each track's end and start: the most common GT id in its last/first k matched frames."""
    if len(matches) == 0:
        return pd.DataFrame(columns=["gid_end", "gid_start"])
    m = matches.sort_values(["trk_id", "frame"])
    mode = lambda s: int(s.mode().iloc[0]) if len(s) else -1  # noqa: E731
    end = m.groupby("trk_id").tail(k).groupby("trk_id").gt_id.agg(mode)
    start = m.groupby("trk_id").head(k).groupby("trk_id").gt_id.agg(mode)
    return pd.DataFrame({"gid_end": end, "gid_start": start})


def label_links(cands: pd.DataFrame, matches: pd.DataFrame, k: int = 10) -> np.ndarray:
    """1 if A's end and C's start are matched to the same GT object, else 0."""
    if len(cands) == 0:
        return np.zeros(0, dtype=int)
    ident = track_identities(matches, k)
    ge = cands.a.map(ident.gid_end).fillna(-1).astype(int).to_numpy()
    gs = cands.c.map(ident.gid_start).fillna(-1).astype(int).to_numpy()
    return ((ge == gs) & (ge >= 0)).astype(int)


def assign(cands: pd.DataFrame, score: str, tau: float) -> pd.DataFrame:
    """One-to-one choice among pairs with score >= tau, maximising the summed log-odds."""
    sel = cands[cands[score] >= tau].drop_duplicates(["a", "c"])
    if len(sel) == 0:
        return sel
    a_ids, ai = np.unique(sel.a.to_numpy(), return_inverse=True)
    c_ids, ci = np.unique(sel.c.to_numpy(), return_inverse=True)
    na = len(a_ids)
    # pairs are local in time and space, so the bipartite graph splits into small components:
    # solve each one exactly instead of one huge assignment problem
    adj = coo_matrix((np.ones(len(sel)), (ai, na + ci)), shape=(na + len(c_ids),) * 2)
    _, comp = connected_components(adj, directed=False)
    pr = sel[score].clip(1e-6, 1 - 1e-6).to_numpy()
    gain = np.log(pr / (1 - pr)) + 50.0  # shift: every real pair beats "no pair"
    keep = np.zeros(len(sel), dtype=bool)
    pair_comp = comp[ai]
    order = np.argsort(pair_comp, kind="stable")
    bounds = np.flatnonzero(np.diff(pair_comp[order])) + 1
    for idx in np.split(order, bounds):
        if len(idx) == 1:
            keep[idx] = True
            continue
        ra, rinv = np.unique(ai[idx], return_inverse=True)
        rc, cinv = np.unique(ci[idx], return_inverse=True)
        cost = np.full((len(ra), len(rc)), 1e6)
        cost[rinv, cinv] = -gain[idx]
        r, c = linear_sum_assignment(cost)
        ok = cost[r, c] < 1e5
        chosen = set(zip(r[ok], c[ok]))
        keep[idx] = [(x, y) in chosen for x, y in zip(rinv, cinv)]
    return sel[keep]


def apply_links(tracks: np.ndarray, pairs: pd.DataFrame) -> np.ndarray:
    """Relabel each C (and its own successors) with the id of the start of its chain."""
    out = tracks.copy()
    if len(pairs) == 0:
        return out
    parent = dict(zip(pairs.c.astype(int), pairs.a.astype(int)))

    def root(x: int) -> int:
        seen = set()
        while x in parent and x not in seen:
            seen.add(x)
            x = parent[x]
        return x

    ids = out[:, ID].astype(int)
    out[:, ID] = [root(i) if i in parent else i for i in ids]
    return out


def verify_pair(backend, tracks: np.ndarray, a: int, c: int, p: LinkParams | None = None) -> dict:
    """Seed the VOS model on A's last clean box and check whether the mask lands on C."""
    p = p or LinkParams()
    ta = tracks[tracks[:, ID] == a]
    tc = tracks[tracks[:, ID] == c]
    ta, tc = ta[np.argsort(ta[:, F])], tc[np.argsort(tc[:, F])]
    seed = ta[-1]
    for r in ta[::-1][:p.seed_lookback]:
        f = r[F]
        others = tracks[(tracks[:, F] == f) & (tracks[:, ID] != a)]
        if len(others) == 0 or iou_matrix(_xyxy(r), _xyxy(others))[0].max() < p.seed_iso_iou:
            seed = r
            break
    seed_f, c_start = int(seed[F]), int(tc[0, F])
    check = tc[:p.check_len]
    last_check = int(check[-1, F])
    n = last_check - seed_f + 1
    info = dict(a=a, c=c, seed_frame=seed_f, frames=0, ioma_c=np.nan, ioma_other=np.nan, confirmed=False)
    if n > p.max_verify_len:
        info["skipped"] = "too_long"
        return info
    c_by_f = {int(r[F]): r for r in check}
    vals_c, vals_o = [], []
    for f, mask, _score in backend.propagate(seed_f, _xyxy(seed)[0], n):
        info["frames"] += 1
        if f in c_by_f:
            others = tracks[(tracks[:, F] == f) & (tracks[:, ID] != c) & (tracks[:, ID] != a)]
            vals_c.append(float(ioma(mask, _xyxy(c_by_f[f]))[0]))
            vals_o.append(float(ioma(mask, _xyxy(others)).max()) if len(others) else 0.0)
        if f >= last_check:
            break
    if len(vals_c) >= 2:
        mc, mo = float(np.median(vals_c)), float(np.median(vals_o))
        info.update(ioma_c=mc, ioma_other=mo, confirmed=bool(mc >= p.confirm_ioma and mc > mo))
    return info

In [ ]:
%%writefile /content/rad_mot/scripts/run_linking.py
#!/usr/bin/env python
"""Method 3 (learned tracklet linking) and method 4 (VOS check of uncertain links) on top of ByteTrack.

For each video: ByteTrack on the given detections (YOLO by default), every (A ended, C born)
pair near in time and space, P(same object) from a gradient-boosted classifier, then
  bytetrack     - the tracker alone
  link_upper    - every true pair merged (a perfect classifier): the ceiling of linking
  link          - method 3: merge pairs with p >= tau_high (no SAM)
  link_verify   - method 4: also verify pairs with tau_low <= p < tau_high with the VOS backend
                  (--backend oracle: masks from GT, an upper bound; --backend sam3: real SAM 3)

No label of a test video is used to choose anything for it. The classifier is trained out of
fold by video (GroupKFold, 5 folds), and tau_high / tau_low are chosen inside each training
fold from inner out-of-fold scores:
  tau_high maximises (correct merges - lam * wrong merges) on the training videos;
  tau_low  is the lowest threshold whose band [tau_low, tau_high) still holds at least
           --band-precision true pairs on the training videos (a verification is worth a call).
At inference the model and both thresholds are fixed; nothing needs labels.

Results per video go to <out>/per_seq/<seq>.csv as soon as it is done (videos already there are
skipped, so an interrupted SAM 3 run resumes), the scored pairs to <out>/pairs/<seq>.csv, and the
totals with bootstrap intervals over videos to <out>/summary.md.

Examples:
    python scripts/run_linking.py --data datasets/FastTracker/train --dets dir --det-dir outputs/dets_yolo \
        --backend oracle --out outputs/link_yolo/oracle
    python scripts/run_linking.py ... --backend sam3 --precision fp16 --frames-on-gpu --max-verify 20 \
        --out outputs/link_yolo/sam3
"""
from __future__ import annotations

import argparse
import glob
import json
import os
import sys
import time

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupKFold

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from rad.config import load_config  # noqa: E402
from rad.dispatch.model import fit_full, train_oof  # noqa: E402
from rad.eval.idsw import clear_match, idf1  # noqa: E402
from rad.io.mot import discover_sequences  # noqa: E402
from rad.link import LINK_FEATURES, LinkParams, apply_links, assign, label_links, link_candidates, verify_pair  # noqa: E402
from rad.pilot.analysis import classify_switches  # noqa: E402
from rad.pipeline import load_detections, load_gt, make_backend, prepare  # noqa: E402

GRID = np.round(np.arange(0.05, 1.0, 0.05), 2)


def metrics(gt: np.ndarray, trk: np.ndarray) -> dict:
    cm = clear_match(gt, trk)
    out = dict(idsw=cm.num_switches, mota=cm.mota, fp=cm.num_fp, fn=cm.num_fn)
    out["idf1"] = idf1(gt, trk)["idf1"]
    types = classify_switches(cm, trk)["type"].value_counts() if cm.num_switches else pd.Series(dtype=int)
    for t in ("fragment", "swap", "takeover"):
        out[t] = int(types.get(t, 0))
    return out


def choose_taus(train: pd.DataFrame, score: str, lam: float, band_precision: float) -> tuple[float, float]:
    best, tau_high = -np.inf, 1.0
    for tau in GRID:
        tp = fp = 0
        for _, g in train.groupby("seq"):
            sel = assign(g, score, tau)
            tp += int(sel.y.sum())
            fp += int(len(sel) - sel.y.sum())
        obj = tp - lam * fp
        if obj > best or (obj == best and tau > tau_high):
            best, tau_high = obj, float(tau)
    tau_low = tau_high
    for t in sorted([t for t in GRID if t < tau_high], reverse=True):
        band = train[(train[score] >= t) & (train[score] < tau_high)]
        if len(band) and band.y.mean() >= band_precision:
            tau_low = float(t)
    return tau_high, tau_low


def nested_scores(lc: pd.DataFrame, lam: float, band_precision: float, kind: str = "hgb") -> tuple[pd.Series, pd.DataFrame]:
    """Out-of-fold P(same object) for every pair, and per-video thresholds chosen without its labels."""
    seqs = lc.seq.to_numpy()
    n_groups = len(np.unique(seqs))
    p = pd.Series(np.nan, index=lc.index)
    taus = []
    gkf = GroupKFold(n_splits=min(5, n_groups))
    for fold, (tr, te) in enumerate(gkf.split(lc, lc.y, seqs)):
        train, test = lc.iloc[tr].copy(), lc.iloc[te]
        inner, _ = train_oof(train, "y", LINK_FEATURES, kind=kind, n_splits=min(4, train.seq.nunique()))
        train["p_in"] = inner.fillna(0.0)
        th, tl = choose_taus(train, "p_in", lam, band_precision)
        if train.y.sum() and train.y.sum() < len(train):
            model = fit_full(train, "y", LINK_FEATURES, kind=kind)
            p.iloc[te] = model.predict_proba(test[LINK_FEATURES].astype(float).to_numpy())[:, 1]
        else:
            p.iloc[te] = 0.0
        for s in np.unique(test.seq):
            taus.append(dict(seq=s, fold=fold, tau_high=th, tau_low=tl))
    return p, pd.DataFrame(taus).set_index("seq")


def bootstrap_sum(x: np.ndarray, n_boot: int = 20000, seed: int = 0) -> tuple[float, float]:
    rng = np.random.default_rng(seed)
    b = x[rng.integers(0, len(x), size=(n_boot, len(x)))].sum(1)
    return float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5))


def summarise(out: str) -> str:
    files = sorted(glob.glob(os.path.join(out, "per_seq", "*.csv")))
    if not files:
        return "no results yet"
    per = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    per.to_csv(os.path.join(out, "all_per_seq.csv"), index=False)
    g = per.groupby("policy", sort=False)
    cols = ["idsw", "fragment", "swap", "takeover", "merges", "merges_wrong", "verify_calls", "verify_confirmed",
            "verify_wrong", "vos_frames", "sec_vos"]
    summ = g[cols].sum()
    summ["videos"] = g.seq.nunique()
    summ["idf1_mean"] = g.idf1.mean()
    summ["mota_mean"] = g.mota.mean()
    summ = summ.reindex([p for p in ("bytetrack", "link_upper", "link", "link_verify") if p in summ.index])
    summ.to_csv(os.path.join(out, "summary.csv"))
    lines = ["# Linking: summary", "", summ.round(3).to_string(), ""]
    piv = per.pivot_table(index="seq", columns="policy", values="idsw")
    for pol in ("link_upper", "link", "link_verify"):
        if pol in piv and "bytetrack" in piv:
            d = (piv.bytetrack - piv[pol]).dropna().to_numpy()
            lo, hi = bootstrap_sum(d)
            lines.append(f"IDSW reduction vs ByteTrack, {pol}: {d.sum():.0f} (95% bootstrap over {len(d)} videos: "
                         f"{lo:.0f} to {hi:.0f}); better on {(d > 0).sum()}, worse on {(d < 0).sum()} videos")
    txt = "\n".join(lines)
    with open(os.path.join(out, "summary.md"), "w") as fh:
        fh.write(txt + "\n")
    return txt


def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--data", required=False)
    ap.add_argument("--config", default=os.path.join(os.path.dirname(__file__), "..", "configs", "default.yaml"))
    ap.add_argument("--dets", choices=["dir", "det", "oracle"], default="dir")
    ap.add_argument("--det-dir", default="outputs/dets_yolo")
    ap.add_argument("--out", required=True)
    ap.add_argument("--backend", choices=["none", "oracle", "sam3"], default="oracle")
    ap.add_argument("--precision", choices=["auto", "fp16"], default="fp16")
    ap.add_argument("--frames-on-gpu", action="store_true")
    ap.add_argument("--max-verify", type=int, default=0, help="most verifications per video, highest p first (0 = all)")
    ap.add_argument("--lam", type=float, default=2.0, help="cost of a wrong merge relative to a correct one")
    ap.add_argument("--band-precision", type=float, default=0.2)
    ap.add_argument("--max-gap-s", type=float, default=3.0)
    ap.add_argument("--radius", type=float, default=5.0)
    ap.add_argument("--run", nargs="*", default=None, help="videos to evaluate, in this order (default: all)")
    ap.add_argument("--summary-only", action="store_true")
    args = ap.parse_args()
    os.makedirs(os.path.join(args.out, "per_seq"), exist_ok=True)
    os.makedirs(os.path.join(args.out, "pairs"), exist_ok=True)
    if args.summary_only:
        print(summarise(args.out))
        return
    if not args.data:
        sys.exit("--data is required")
    cfg = load_config(args.config)
    lp = LinkParams(max_gap_s=args.max_gap_s, radius=args.radius)
    ign = dict(ignore_classes=cfg["gt_ignore_classes"], ignore_ioa=cfg["det_ignore_ioa"])

    t0 = time.perf_counter()
    data, parts = {}, []
    for s in discover_sequences(args.data):
        _, gt_vis = load_gt(s, **ign)
        det = load_detections(s, args.dets, args.det_dir, cfg["oracle_dets"], gt_vis=gt_vis, **ign)
        sd = prepare(s, det, cfg["bytetrack"], cfg["candidates"], **ign)
        sd.cands = None  # RAD's match candidates are not used here
        t = time.perf_counter()
        lc = link_candidates(sd.tracks, sd.size, s.fps or 30.0, lp)
        lc["seq"] = s.name
        lc["y"] = label_links(lc, sd.cm.matches, lp.label_k)
        data[s.name] = (sd, time.perf_counter() - t)
        parts.append(lc)
        print(f"[prepare] {s.name}: {len(sd.tracks):,} track boxes, {len(lc):,} pairs, {int(lc.y.sum())} true",
              flush=True)
    lc_all = pd.concat(parts, ignore_index=True)
    print(f"[prepare] {len(data)} videos, {len(lc_all):,} pairs, {int(lc_all.y.sum())} true "
          f"({time.perf_counter() - t0:.0f}s)", flush=True)

    t = time.perf_counter()
    lc_all["p"], taus = nested_scores(lc_all, args.lam, args.band_precision)
    ok = lc_all.p.notna()
    mm = dict(n=int(len(lc_all)), positives=int(lc_all.y.sum()), seconds=round(time.perf_counter() - t, 1))
    if ok.sum() and lc_all.y[ok].nunique() == 2:
        mm.update(roc_auc=float(roc_auc_score(lc_all.y[ok], lc_all.p[ok])),
                  pr_auc=float(average_precision_score(lc_all.y[ok], lc_all.p[ok])))
    mm["taus"] = taus.reset_index().to_dict("records")
    with open(os.path.join(args.out, "model_metrics.json"), "w") as fh:
        json.dump(mm, fh, indent=1)
    print("[model]", {k: v for k, v in mm.items() if k != "taus"}, flush=True)
    print(taus.to_string(), flush=True)

    kw = {}
    if args.backend == "oracle":
        kw = dict(cfg["oracle_backend"])
    elif args.backend == "sam3":
        kw = dict(cfg["sam3_backend"])
        kw["precision"] = args.precision
        if args.frames_on_gpu:
            kw["offload_video_to_cpu"] = False
    order = args.run or list(data)
    done = {os.path.splitext(os.path.basename(f))[0] for f in glob.glob(os.path.join(args.out, "per_seq", "*.csv"))}
    for name in order:
        if name in done:
            print(f"[skip] {name}: already in per_seq/")
            continue
        sd, sec_cands = data[name]
        if args.backend == "sam3" and not sd.seq.frame_paths():
            print(f"[skip] {name}: no img1/ frames")
            continue
        lc = lc_all[lc_all.seq == name].copy()
        th, tl = float(taus.loc[name, "tau_high"]), float(taus.loc[name, "tau_low"])
        common = dict(seq=name, frames=sd.length, n_pairs=len(lc), n_true=int(lc.y.sum()), tau_high=th, tau_low=tl,
                      sec_link=sec_cands)
        rows = []
        zero = dict(merges=0, merges_wrong=0, verify_calls=0, verify_confirmed=0, verify_wrong=0, vos_frames=0, sec_vos=0.0)
        rows.append(dict(**common, policy="bytetrack", **metrics(sd.gt, sd.tracks), **zero))
        up = assign(lc.assign(yf=lc.y.astype(float)), "yf", 0.5)
        rows.append(dict(**common, policy="link_upper", **metrics(sd.gt, apply_links(sd.tracks, up)),
                         **{**zero, "merges": len(up)}))
        sel = assign(lc, "p", th)
        rows.append(dict(**common, policy="link", **metrics(sd.gt, apply_links(sd.tracks, sel)),
                         **{**zero, "merges": len(sel), "merges_wrong": int((sel.y == 0).sum())}))
        lc["decision"] = "none"
        lc.loc[lc.index.isin(sel.index), "decision"] = "merge"
        if args.backend != "none":
            band = lc[(lc.p >= tl) & (lc.p < th) & ~lc.a.isin(sel.a) & ~lc.c.isin(sel.c)].sort_values("p", ascending=False)
            if args.max_verify:
                band = band.head(args.max_verify)
            be = make_backend(args.backend, sd, **kw)
            infos, t = [], time.perf_counter()
            for r in band.itertuples():
                info = verify_pair(be, sd.tracks, int(r.a), int(r.c), lp)
                info["idx"] = r.Index
                infos.append(info)
            sec_vos = time.perf_counter() - t
            conf_idx = [i["idx"] for i in infos if i["confirmed"]]
            lc.loc[band.index, "decision"] = "verify_rejected"
            lc.loc[conf_idx, "decision"] = "verify_confirmed"
            extra = assign(lc.loc[conf_idx], "p", 0.0) if conf_idx else lc.iloc[:0]
            final = pd.concat([sel, extra])
            rows.append(dict(**common, policy="link_verify", **metrics(sd.gt, apply_links(sd.tracks, final)),
                             merges=len(final), merges_wrong=int((final.y == 0).sum()), verify_calls=len(infos),
                             verify_confirmed=len(conf_idx), verify_wrong=int((lc.loc[conf_idx].y == 0).sum()),
                             vos_frames=int(sum(i["frames"] for i in infos)), sec_vos=sec_vos))
            if infos:
                pd.DataFrame(infos).to_csv(os.path.join(args.out, "pairs", f"{name}_verify.csv"), index=False)
            del be
        lc.to_csv(os.path.join(args.out, "pairs", f"{name}.csv"), index=False)
        pd.DataFrame(rows).to_csv(os.path.join(args.out, "per_seq", f"{name}.csv"), index=False)
        r = {x["policy"]: x for x in rows}
        msg = f"[done] {name}: IDSW ByteTrack {r['bytetrack']['idsw']}, upper {r['link_upper']['idsw']}, " \
              f"link {r['link']['idsw']} ({r['link']['merges']} merges, {r['link']['merges_wrong']} wrong)"
        if "link_verify" in r:
            v = r["link_verify"]
            msg += f", link+verify {v['idsw']} ({v['verify_calls']} checks, {v['verify_confirmed']} confirmed, " \
                   f"{v['verify_wrong']} wrong, {v['sec_vos'] / 60:.1f} min)"
        print(msg, flush=True)
        summarise(args.out)
    print(summarise(args.out))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/rad_mot/scripts/run_trackers.py
#!/usr/bin/env python
"""Pure multi-object trackers on the same detections, same GT, same metrics.

Trackers (all class-agnostic, all fed the same detection file per video):
  bytetrack    - this repo's ByteTrack (the baseline of every other experiment)
  ocsort       - official OC-SORT (Cao et al., CVPR 2023), default parameters of its repo
                 (det_thresh 0.6, IoU 0.3, min_hits 3, delta_t 3, inertia 0.2, no BYTE)
  ocsort_byte  - the same with its BYTE second association (use_byte=True)
With --link, each tracker is also run with method 3 on top (learned tracklet linking, no SAM):
the link model is trained on that tracker's own tracks, out of fold by video, thresholds
chosen inside the training folds, exactly as in scripts/run_linking.py.

Metrics per video and in total: IDSW and its types (fragment / swap / takeover), IDF1, MOTA
from this repo's CLEAR-MOT, and HOTA / DetA / AssA from TrackEval when it is installed.
Speed: seconds of the tracker alone; with --det-sec-per-frame (default: YOLO11x at 1280 px on
a T4, measured 10/11/2026) also the frame rate of detector + tracker.

OC-SORT is the official code (MIT licence): clone it and pass its folder with --ocsort-dir
    git clone https://github.com/noahcao/OC_SORT /content/OC_SORT

Example:
    python scripts/run_trackers.py --data datasets/FastTracker/train --dets dir --det-dir outputs/dets_yolo \
        --ocsort-dir /content/OC_SORT --link --out outputs/trackers_yolo
"""
from __future__ import annotations

import argparse
import os
import sys
import time
from dataclasses import asdict

import numpy as np
import pandas as pd

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, os.path.dirname(HERE))
sys.path.insert(0, HERE)
from rad.config import load_config  # noqa: E402
from rad.eval.idsw import clear_match  # noqa: E402
from rad.eval.trackeval_runner import available as trackeval_available, run_trackeval  # noqa: E402
from rad.io.mot import F, H, S, W, X, Y, discover_sequences, write_mot_txt  # noqa: E402
from rad.link import LinkParams, apply_links, assign, label_links, link_candidates  # noqa: E402
from rad.pipeline import load_detections, load_gt  # noqa: E402
from rad.tracker.bytetrack import ByteTrackParams, run_bytetrack  # noqa: E402
from run_linking import bootstrap_sum, metrics, nested_scores  # noqa: E402

YOLO11X_SEC_PER_FRAME = 1776.2 / 19480  # 12 FastTracker videos, 1280 px, fp16, T4 (from file times)


def run_ocsort(det: np.ndarray, length: int, size: tuple, fps: float, ocsort_cls, use_byte: bool) -> np.ndarray:
    """Official OC-SORT, frame by frame; max_age scaled with the frame rate like ByteTrack's buffer."""
    w, h = size
    trk = ocsort_cls(det_thresh=0.6, max_age=max(1, int(round(30 * fps / 30.0))), min_hits=3, iou_threshold=0.3,
                     delta_t=3, asso_func="iou", inertia=0.2, use_byte=use_byte)
    frames = det[:, F].astype(int)
    order = np.argsort(frames, kind="stable")
    det, frames = det[order], frames[order]
    starts = np.searchsorted(frames, np.arange(1, length + 2))
    rows = []
    for f in range(1, length + 1):
        d = det[starts[f - 1]:starts[f]]
        arr = (np.stack([d[:, X], d[:, Y], d[:, X] + d[:, W], d[:, Y] + d[:, H], d[:, S]], 1)
               if len(d) else np.empty((0, 5)))
        for x1, y1, x2, y2, tid in trk.update(arr, (h, w), (h, w)):
            rows.append([f, int(tid), x1, y1, x2 - x1, y2 - y1, 1.0, -1, -1])
    return np.asarray(rows, dtype=float) if rows else np.zeros((0, 9))


def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--data", required=True)
    ap.add_argument("--config", default=os.path.join(HERE, "..", "configs", "default.yaml"))
    ap.add_argument("--dets", choices=["dir", "det", "oracle"], default="dir")
    ap.add_argument("--det-dir", default="outputs/dets_yolo")
    ap.add_argument("--trackers", nargs="*", default=["bytetrack", "ocsort", "ocsort_byte"])
    ap.add_argument("--ocsort-dir", default="OC_SORT", help="folder of a clone of github.com/noahcao/OC_SORT")
    ap.add_argument("--link", action="store_true", help="also run each tracker + method 3 (learned linking)")
    ap.add_argument("--lam", type=float, default=2.0)
    ap.add_argument("--det-sec-per-frame", type=float, default=YOLO11X_SEC_PER_FRAME,
                    help="detector time per frame for the pipeline frame rate (0: tracker only)")
    ap.add_argument("--out", required=True)
    args = ap.parse_args()
    cfg = load_config(args.config)
    ign = dict(ignore_classes=cfg["gt_ignore_classes"], ignore_ioa=cfg["det_ignore_ioa"])
    ocsort_cls = None
    if any(t.startswith("ocsort") for t in args.trackers):
        sys.path.insert(0, os.path.abspath(args.ocsort_dir))
        try:
            from trackers.ocsort_tracker.ocsort import OCSort as ocsort_cls  # noqa: N813
        except ImportError as e:
            sys.exit(f"OC-SORT not found in {args.ocsort_dir} ({e}); clone github.com/noahcao/OC_SORT there")
    os.makedirs(args.out, exist_ok=True)

    seqs, rows, tracks_by = [], [], {}
    for s in discover_sequences(args.data):
        gt, gt_vis = load_gt(s, **ign)
        det = load_detections(s, args.dets, args.det_dir, cfg["oracle_dets"], gt_vis=gt_vis, **ign)
        length = int(s.length or max(gt[:, F].max() if len(gt) else 0, det[:, F].max() if len(det) else 0))
        size = (s.width or 1920, s.height or 1080)
        fps = float(s.fps or 30.0)
        seqs.append(dict(name=s.name, gt=gt, length=length, size=size, fps=fps))
        for name in args.trackers:
            t = time.perf_counter()
            if name == "bytetrack":
                trk, _ = run_bytetrack(det, length, ByteTrackParams(**{**asdict(cfg["bytetrack"]), "frame_rate": fps}))
            elif name in ("ocsort", "ocsort_byte"):
                trk = run_ocsort(det, length, size, fps, ocsort_cls, use_byte=name == "ocsort_byte")
            else:
                sys.exit(f"unknown tracker {name}")
            sec = time.perf_counter() - t
            tracks_by[(name, s.name)] = trk
            write_mot_txt(os.path.join(args.out, "tracks", name, f"{s.name}.txt"), trk)
            rows.append(dict(seq=s.name, tracker=name, frames=length, sec_tracker=sec, **metrics(gt, trk)))
            print(f"[{s.name}] {name}: IDSW {rows[-1]['idsw']}, IDF1 {rows[-1]['idf1']:.3f}, "
                  f"{length / max(sec, 1e-9):.0f} frames/s", flush=True)

    if args.link:
        lp = LinkParams()
        for name in args.trackers:
            parts = []
            for sq in seqs:
                trk = tracks_by[(name, sq["name"])]
                t = time.perf_counter()
                lc = link_candidates(trk, sq["size"], sq["fps"], lp)
                lc["seq"] = sq["name"]
                lc["sec"] = time.perf_counter() - t
                lc["y"] = label_links(lc, clear_match(sq["gt"], trk).matches, lp.label_k)
                parts.append(lc)
            lc_all = pd.concat(parts, ignore_index=True)
            lc_all["p"], taus = nested_scores(lc_all, args.lam, 0.2)
            for sq in seqs:
                trk = tracks_by[(name, sq["name"])]
                lc = lc_all[lc_all.seq == sq["name"]]
                t = time.perf_counter()
                sel = assign(lc, "p", float(taus.loc[sq["name"], "tau_high"]))
                linked = apply_links(trk, sel)
                sec = time.perf_counter() - t + (float(lc["sec"].iloc[0]) if len(lc) else 0.0)
                base = next(r for r in rows if r["seq"] == sq["name"] and r["tracker"] == name)
                tracks_by[(name + "+link", sq["name"])] = linked
                write_mot_txt(os.path.join(args.out, "tracks", name + "+link", f"{sq['name']}.txt"), linked)
                rows.append(dict(seq=sq["name"], tracker=name + "+link", frames=sq["length"],
                                 sec_tracker=base["sec_tracker"] + sec, merges=len(sel),
                                 merges_wrong=int((sel.y == 0).sum()), **metrics(sq["gt"], linked)))
            print(f"[link] {name}: {int(lc_all.y.sum())} true pairs of {len(lc_all)}", flush=True)

    per = pd.DataFrame(rows)
    variants = list(dict.fromkeys(per.tracker))
    if trackeval_available():
        for v in variants:
            te = run_trackeval({sq["name"]: sq["gt"] for sq in seqs},
                               {sq["name"]: tracks_by[(v, sq["name"])] for sq in seqs},
                               {sq["name"]: sq["length"] for sq in seqs}, tracker_name=v.replace("+", "_"))
            for k in ("HOTA", "DetA", "AssA"):
                per.loc[per.tracker == v, k] = per.loc[per.tracker == v, "seq"].map(lambda s: te[s][k])
                per.loc[per.tracker == v, k + "_all"] = te["COMBINED"][k]
    else:
        print("[trackeval] not installed: HOTA skipped (pip install git+https://github.com/JonathonLuiten/TrackEval)")
    per.to_csv(os.path.join(args.out, "per_seq.csv"), index=False)

    g = per.groupby("tracker", sort=False)
    summ = g[["idsw", "fragment", "swap", "takeover"]].sum()
    summ["idf1_mean"] = g.idf1.mean()
    summ["mota_mean"] = g.mota.mean()
    for k in ("HOTA", "DetA", "AssA"):
        if k + "_all" in per:
            summ[k] = g[k + "_all"].first()
    summ["tracker_fps"] = g.frames.sum() / g.sec_tracker.sum()
    if args.det_sec_per_frame > 0:
        summ["pipeline_fps"] = g.frames.sum() / (g.sec_tracker.sum() + args.det_sec_per_frame * g.frames.sum())
    summ = summ.reindex(variants)
    summ.to_csv(os.path.join(args.out, "summary.csv"))
    lines = ["# Trackers on the same detections", "", summ.round(3).to_string(), ""]
    piv = per.pivot_table(index="seq", columns="tracker", values="idsw")
    if "bytetrack" in piv:
        for v in variants:
            if v != "bytetrack":
                d = (piv.bytetrack - piv[v]).to_numpy()
                lo, hi = bootstrap_sum(d)
                lines.append(f"IDSW reduction vs ByteTrack, {v}: {d.sum():.0f} (95% bootstrap: {lo:.0f} to {hi:.0f}); "
                             f"better on {(d > 0).sum()}, worse on {(d < 0).sum()} of {len(d)} videos")
    txt = "\n".join(lines)
    with open(os.path.join(args.out, "summary.md"), "w") as fh:
        fh.write(txt + "\n")
    print(txt)


if __name__ == "__main__":
    main()

## 3. Token Hugging Face và nhãn (chỉ nhãn, không tải ảnh)

In [ ]:
from huggingface_hub import get_token, login
if not get_token():
    try:
        from google.colab import userdata
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    except Exception:
        pass
if not get_token():
    login()
!python scripts/download_fasttracker.py --out $DS --mode gt

## 4. Chạy và chấm điểm

In [ ]:
os.makedirs(OUT, exist_ok=True)
!python -W ignore -u scripts/run_trackers.py --data $DATA --dets dir --det-dir $DETS --ocsort-dir /content/OC_SORT --link --out $OUT 2>&1 | tee $OUT/run.log

## 5. Kết quả

In [ ]:
print(open(f'{OUT}/summary.md').read() if os.path.exists(f'{OUT}/summary.md') else 'chưa có kết quả: xem log ở ô trên')